# Replication Notebook — ICU Example, Final Models (Step 2 of 3)

This notebook reproduces the main‐text and appendix ICU results for the four parametric / tree‐based models from the paper

*“The choice of reference group can reverse conclusions in the Oaxaca–Blinder decomposition.”*

This is **Step 2** of the replication pipeline. It loads `ICU_clean.csv` (produced by `construct_ICU_data.ipynb`, Step 1), builds 139 clinical subsets, and runs four models with bootstrap inference. TabPFN is reproduced separately in `icu_139_tabpfn_local.ipynb` (Step 3).

> See `construct_ICU_data.ipynb` for environment setup, data access, and full pipeline overview.

## What this notebook produces

- **Part A** — HR‐quartile‐2 coefficient table (`hrq2_bootstrap.csv`): explained / unexplained components under both reference choices, with bootstrap SEs and significance stars, for all four models. Reproduces the four parametric / tree rows of Table 1.
- **Part A — Figures** — fitted‐mortality‐by‐HR plot (`Figures/hr_quartile2_mortality_lines.pdf`, **Figure 1 Left**) and 0/1 mortality histogram by gender (`Figures/hr_quartile2_mortality_y_hist.pdf`, **Appendix Figure 2**), both within HR quartile 2.
- **Part B** — Bootstrap inference on all 139 subsets (`all_subsets_bootstrap.csv`).
- **Three flip‐summary tables** (`Model | Total flips | 10% | 5% | 1%`) — non‐TabPFN rows of Table 1 and Appendix Tables 3, 4, 5:
  - Subset‐wise (`flip_summary_subsetwise.csv`)
  - Explained‐component (`flip_summary_explained.csv`)
  - Unexplained‐component (`flip_summary_unexplained.csv`)
  - Hearth Rate Quartile 2 

**Sign‐flip definition.** A component (explained or unexplained) flips if its sign differs between men‐reference and women‐reference. Significant at level α: at least one reference has p < α.

## Models and subsets

- Covariates: `[Age, ICUType, HR, NIMAP, Temp, Urine]` (base set).
- 139 subsets: full cohort, ICU types, age deciles, HR / MAP / temp / urine / SAPS quartiles, clinical thresholds, 50 random 50 %‐subsamples, 50 random 30 %‐subsamples.
- Models: linear (statsmodels OLS), logistic (sklearn), neural net (sklearn `MLPClassifier`, hidden = `(32,16,8,4)`, adam, max_iter = 2000), XGBoost (`n_estimators=500, max_depth=3, lr=0.05`).
- Bootstrap: pair‐bootstrap stratified by group, **B = 1000** replications, parallelised with `joblib`.

## How to run

Make sure `ICU_clean.csv` is in the same folder (produced by running Step 1 first). Then execute every cell top to bottom. Outputs are written next to the notebook.


In [4]:
# ============================================================
# SECTION 0 — Imports
# ============================================================
import os, warnings, uuid
from typing import List, Dict, Any, Callable

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

from scipy.stats import norm
from joblib import Parallel, delayed

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    XGBClassifier = None

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 250)


In [ ]:
# ============================================================
# SECTION 1 — Config
# ============================================================
CONFIG = {
    "random_state": 51,
    "high_group_value": 0,       # 0 = men (reference direction)
    "min_subset_size": 100,
    "B_linear": 1000,
    "B_ml": 1000,                # set to lower (e.g., 300) if too slow
    "alpha_default": 0.05,
    "bootstrap_n_jobs": -1,
    "n_random_subsamples": 50,
    "random_subsample_fracs": [0.50, 0.30],
}

Y_COL     = "In-hospital_death"
GROUP_COL = "female"

X_COLS = ["Age", "ICUType", "HR", "NIMAP", "Temp", "Urine"] 

print("X_COLS:", X_COLS)


In [ ]:
# ============================================================
# SECTION 2 — Load Data (base ICU_clean.csv — not extended)
#   Matches reference notebooks: dropna applied globally BEFORE
#   constructing any subsets (including the random ones). This is
#   what the reference icu_analysis*.ipynb notebooks do.
# ============================================================
DATA_PATH = "ICU_clean.csv"
df_raw = pd.read_csv(DATA_PATH)
print("Loaded:", df_raw.shape)

needed = X_COLS + [Y_COL, GROUP_COL]
missing = [c for c in needed if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing columns in {DATA_PATH}: {missing}")

df_clean = df_raw.dropna(subset=needed).copy()


In [ ]:
# ============================================================
# SECTION 3 — Construct 139 clinical subsets
#   Uses legacy np.random.seed + np.random.choice for the 100 random
#   subsamples so results match icu_analysis.ipynb / icu_analysis_nonlinear.ipynb.
# ============================================================
def make_quartile_subsets(df, col, label_prefix):
    q = pd.qcut(df[col], q=4, labels=False, duplicates="drop")
    return [(f"{label_prefix} quartile {int(k)}", df[q == k]) for k in sorted(q.dropna().unique())]

def make_decile_subsets(df, col, label_prefix):
    d = pd.qcut(df[col], q=10, labels=False, duplicates="drop")
    return [(f"{label_prefix} decile {int(k)}", df[d == k]) for k in sorted(d.dropna().unique())]

subsets_raw = []
subsets_raw.append(("All Patients", df_clean))

for t in sorted(df_clean["ICUType"].dropna().unique()):
    subsets_raw.append((f"ICUType {t}", df_clean[df_clean["ICUType"] == t]))

subsets_raw += make_decile_subsets(df_clean, "Age", "Age")
for col, pfx in [("HR", "HR"), ("NIMAP", "MAP"), ("Temp", "Temp"),
                 ("Urine", "Urine"), ("SAPS-I", "SAPS")]:
    if col in df_clean.columns:
        subsets_raw += make_quartile_subsets(df_clean, col, pfx)

subsets_raw.append(("HR > 100",    df_clean[df_clean["HR"] > 100]))
subsets_raw.append(("MAP < 65",    df_clean[df_clean["NIMAP"] < 65]))
subsets_raw.append(("Temp > 38C",  df_clean[df_clean["Temp"] > 38]))
subsets_raw.append(("Urine > 1000", df_clean[df_clean["Urine"] > 1000]))

# 50 random 50% + 50 random 30% subsamples
# LEGACY RNG (np.random.seed + np.random.choice) to match icu_analysis*.ipynb exactly.
np.random.seed(CONFIG["random_state"])
for i in range(1, CONFIG["n_random_subsamples"] + 1):
    idx = np.random.choice(df_clean.index, size=int(0.5 * len(df_clean)), replace=False)
    subsets_raw.append((f"Random 50% #{i}", df_clean.loc[idx]))
for i in range(1, CONFIG["n_random_subsamples"] + 1):
    idx = np.random.choice(df_clean.index, size=int(0.3 * len(df_clean)), replace=False)
    subsets_raw.append((f"Random 30% #{i}", df_clean.loc[idx]))

# Drop NaNs on needed cols and enforce min size
subsets = []
for label, sub in subsets_raw:
    sub = sub.dropna(subset=X_COLS + [Y_COL, GROUP_COL]).copy()
    if len(sub) >= CONFIG["min_subset_size"]:
        subsets.append((label, sub))

subset_dict = {lab: sub for lab, sub in subsets}
print(f"Constructed {len(subsets)} subsets (n >= {CONFIG['min_subset_size']}).")


In [8]:
# ============================================================
# SECTION 4 — OBD helpers, bootstrap, model builders
# ============================================================
def add_const(X):
    return sm.add_constant(X, has_constant="add")

def oaxaca_twofold_fixed_direction(df, y_col, x_cols, group_col, high_group_value=0):
    high = df[df[group_col] == high_group_value]
    low  = df[df[group_col] != high_group_value]
    if high.empty or low.empty:
        return {k: np.nan for k in ["difference", "explained_highref", "unexplained_highref",
                                     "explained_lowref", "unexplained_lowref"]}
    Xh = add_const(high[x_cols]); yh = high[y_col]
    Xl = add_const(low[x_cols]);  yl = low[y_col]
    bh = sm.OLS(yh, Xh, missing="drop").fit().params
    bl = sm.OLS(yl, Xl, missing="drop").fit().params
    mh, ml = Xh.mean(), Xl.mean()
    diff = float(yh.mean() - yl.mean())
    return {
        "difference": diff,
        "explained_highref":   float((mh - ml) @ bh),
        "unexplained_highref": float(ml @ (bh - bl)),
        "explained_lowref":    float((mh - ml) @ bl),
        "unexplained_lowref":  float(mh @ (bh - bl)),
    }

def bootstrap_oaxaca_linear(df, y_col, x_cols, group_col, B, random_state=0, n_jobs=-1):
    rng = np.random.default_rng(random_state)
    base = oaxaca_twofold_fixed_direction(df, y_col, x_cols, group_col, 0)
    g0 = df[df[group_col] == 0]; g1 = df[df[group_col] != 0]
    n0, n1 = len(g0), len(g1)
    seeds = rng.integers(1_000_000_000, size=B)
    def one(seed):
        r = np.random.default_rng(int(seed))
        bd = pd.concat([
            g0.sample(n0, replace=True, random_state=int(r.integers(1e9))),
            g1.sample(n1, replace=True, random_state=int(r.integers(1e9))),
        ])
        try:
            return oaxaca_twofold_fixed_direction(bd, y_col, x_cols, group_col, 0)
        except Exception:
            return None
    draws = Parallel(n_jobs=n_jobs, backend="loky")(delayed(one)(s) for s in seeds)
    draws = [d for d in draws if d is not None]
    def col(k): return np.array([d[k] for d in draws], dtype=float)
    def se(a): return float(np.std(a, ddof=1)) if len(a) > 1 else np.nan
    return {
        "difference": base["difference"], "difference_se": se(col("difference")),
        "explained_menref":   base["explained_highref"],   "explained_menref_se":   se(col("explained_highref")),
        "unexplained_menref": base["unexplained_highref"], "unexplained_menref_se": se(col("unexplained_highref")),
        "explained_womenref":   base["explained_lowref"],   "explained_womenref_se":   se(col("explained_lowref")),
        "unexplained_womenref": base["unexplained_lowref"], "unexplained_womenref_se": se(col("unexplained_lowref")),
    }


def predict_positive_class(model, X):
    if hasattr(model, "predict_proba"):
        p = np.asarray(model.predict_proba(X))
        if p.ndim == 2 and p.shape[1] >= 2:
            return p[:, 1].astype(float)
        return p.ravel().astype(float)
    return np.asarray(model.predict(X), dtype=float).ravel()

def nonlinear_oaxaca(df, y_col, x_cols, group_col, model_builder, high_group_value=0):
    high = df[df[group_col] == high_group_value]
    low  = df[df[group_col] != high_group_value]
    if high.empty or low.empty:
        return {k: np.nan for k in ["difference", "explained_highref", "unexplained_highref",
                                     "explained_lowref", "unexplained_lowref"]}
    Xh, yh = high[x_cols], high[y_col]
    Xl, yl = low[x_cols],  low[y_col]
    mh_model = model_builder(); mh_model.fit(Xh, yh)
    ml_model = model_builder(); ml_model.fit(Xl, yl)
    mean_h = float(yh.mean()); mean_l = float(yl.mean())
    M_l_h = float(predict_positive_class(mh_model, Xl).mean())
    M_h_l = float(predict_positive_class(ml_model, Xh).mean())
    return {
        "difference": mean_h - mean_l,
        "explained_highref":   mean_h - M_l_h,
        "unexplained_highref": M_l_h - mean_l,
        "explained_lowref":    M_h_l - mean_l,
        "unexplained_lowref":  mean_h - M_h_l,
    }

def bootstrap_nonlinear(df, y_col, x_cols, group_col, model_builder, B, random_state, n_jobs=-1):
    rng = np.random.default_rng(random_state)
    base = nonlinear_oaxaca(df, y_col, x_cols, group_col, model_builder, 0)
    g0 = df[df[group_col] == 0]; g1 = df[df[group_col] != 0]
    n0, n1 = len(g0), len(g1)
    seeds = rng.integers(1_000_000_000, size=B)
    def one(seed):
        r = np.random.default_rng(int(seed))
        bd = pd.concat([
            g0.sample(n0, replace=True, random_state=int(r.integers(1e9))),
            g1.sample(n1, replace=True, random_state=int(r.integers(1e9))),
        ])
        try:
            return nonlinear_oaxaca(bd, y_col, x_cols, group_col, model_builder, 0)
        except Exception:
            return None
    draws = Parallel(n_jobs=n_jobs, backend="loky")(delayed(one)(s) for s in seeds)
    draws = [d for d in draws if d is not None]
    def col(k): return np.array([d[k] for d in draws], dtype=float)
    def se(a): return float(np.std(a, ddof=1)) if len(a) > 1 else np.nan
    return {
        "difference": base["difference"], "difference_se": se(col("difference")),
        "explained_menref":   base["explained_highref"],   "explained_menref_se":   se(col("explained_highref")),
        "unexplained_menref": base["unexplained_highref"], "unexplained_menref_se": se(col("unexplained_highref")),
        "explained_womenref":   base["explained_lowref"],   "explained_womenref_se":   se(col("explained_lowref")),
        "unexplained_womenref": base["unexplained_lowref"], "unexplained_womenref_se": se(col("unexplained_lowref")),
    }

# Model builders
def build_logistic():
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000, random_state=0))])

def build_nn():
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", MLPClassifier(hidden_layer_sizes=(32, 16),
                                           activation="relu", alpha=1e-4,
                                           max_iter=2000, solver="adam",
                                           random_state=0))])

def build_xgb():
    return XGBClassifier(n_estimators=250, max_depth=3, learning_rate=0.05,
                         subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                         eval_metric="logloss", random_state=0, n_jobs=1,
                         tree_method="hist", verbosity=0)

ML_BUILDERS = {"logistic": build_logistic, "nn": build_nn}
if XGBOOST_AVAILABLE:
    ML_BUILDERS["xgboost"] = build_xgb

def pval(est, se):
    if pd.isna(est) or pd.isna(se) or se <= 0:
        return np.nan
    return 2 * (1 - norm.cdf(abs(est / se)))

def stars(p):
    if pd.isna(p): return ""
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return ""

def build_boot_row(model_name, subset_label, n, out):
    row = {"model": model_name, "subset": subset_label, "n": n}
    for comp in ["difference", "explained_menref", "unexplained_menref",
                 "explained_womenref", "unexplained_womenref"]:
        est = out[comp]; se = out[f"{comp}_se"]; p = pval(est, se)
        row[comp] = est
        row[f"{comp}_se"] = se
        row[f"{comp}_pvalue"] = p
        row[f"{comp}_stars"] = stars(p)
    return row


In [ ]:
# ============================================================
# PART A — HR Quartile 2 only: coef(SE) stars table for all 4 models
# ============================================================
TARGET = "HR quartile 2"
if TARGET not in subset_dict:
    raise ValueError(f"{TARGET} not in subsets. Available: {list(subset_dict)[:5]}...")

sub = subset_dict[TARGET]
print(f"{TARGET}: n={len(sub)}")

hrq2_rows = []

# linear
print("  linear bootstrap...")
out = bootstrap_oaxaca_linear(sub, Y_COL, X_COLS, GROUP_COL,
                              B=CONFIG["B_linear"], random_state=0,
                              n_jobs=CONFIG["bootstrap_n_jobs"])
hrq2_rows.append(build_boot_row("linear", TARGET, len(sub), out))

for mname, mbuilder in ML_BUILDERS.items():
    print(f"  {mname} bootstrap...")
    out = bootstrap_nonlinear(sub, Y_COL, X_COLS, GROUP_COL, mbuilder,
                              B=CONFIG["B_ml"], random_state=1000,
                              n_jobs=CONFIG["bootstrap_n_jobs"])
    hrq2_rows.append(build_boot_row(mname, TARGET, len(sub), out))

hrq2_df = pd.DataFrame(hrq2_rows)
hrq2_df.to_csv("hrq2_bootstrap.csv", index=False)

# Formatted coef(SE) stars table
def fmt(est, se, p):
    if pd.isna(est) or pd.isna(se):
        return ""
    return f"{est:.3f}{stars(p)}\n({se:.3f})"

fmt_df = hrq2_df.copy()
fmt_df["Gap (M-F)"]               = [fmt(e, s, p) for e, s, p in zip(fmt_df["difference"], fmt_df["difference_se"], fmt_df["difference_pvalue"])]
fmt_df["Explained (Women ref)"]   = [fmt(e, s, p) for e, s, p in zip(fmt_df["explained_womenref"], fmt_df["explained_womenref_se"], fmt_df["explained_womenref_pvalue"])]
fmt_df["Unexplained (Women ref)"] = [fmt(e, s, p) for e, s, p in zip(fmt_df["unexplained_womenref"], fmt_df["unexplained_womenref_se"], fmt_df["unexplained_womenref_pvalue"])]
fmt_df["Explained (Men ref)"]     = [fmt(e, s, p) for e, s, p in zip(fmt_df["explained_menref"], fmt_df["explained_menref_se"], fmt_df["explained_menref_pvalue"])]
fmt_df["Unexplained (Men ref)"]   = [fmt(e, s, p) for e, s, p in zip(fmt_df["unexplained_menref"], fmt_df["unexplained_menref_se"], fmt_df["unexplained_menref_pvalue"])]

MODEL_ORDER = ["linear", "logistic", "nn", "xgboost"]
fmt_df["_mo"] = fmt_df["model"].apply(lambda m: MODEL_ORDER.index(m) if m in MODEL_ORDER else 999)
nice_hrq2 = fmt_df.sort_values("_mo")[[
    "subset", "model", "n", "Gap (M-F)",
    "Explained (Women ref)", "Unexplained (Women ref)",
    "Explained (Men ref)",   "Unexplained (Men ref)"
]].reset_index(drop=True)

display(nice_hrq2)
nice_hrq2.to_csv("hrq2_table_origianlnnxgboost.csv", index=False)


## Part A (figures) — Figure 1 (Left) and Appendix Figure 2

Two figures supporting the HR-quartile-2 results above:
- **Figure 1 (Left)** — fitted in-hospital mortality vs. admission heart rate, separately for men and women, holding other covariates at their HR-quartile-2 means.
- **Appendix Figure 2** — histogram of in-hospital mortality (0/1) by gender within HR quartile 2.

Both saved to `Figures/`.


In [ ]:
# ============================================================
# PART A (figures) — Figure 1 (Left) and Appendix Figure 2
# ============================================================
COLOR_MEN   = "#1f77b4"  # blue
COLOR_WOMEN = "#ff7f0e"  # orange


def plot_hr_quartile2_lines(
    df_clean, x_cols, y_col="In-hospital_death", female_col="female",
    target_bin=2, savepath=None,
):
    """Figure 1 (Left): fitted mortality vs. HR by gender, HR quartile 2."""
    df = df_clean.copy()
    df["hr_bin"] = pd.qcut(df["HR"], 4, labels=False, duplicates="drop")
    sub = df[df["hr_bin"] == target_bin].copy()
    print("HR quartile 2 size:", len(sub))

    men   = sub[sub[female_col] == 0].copy()
    women = sub[sub[female_col] == 1].copy()

    Xm, ym = add_const(men[x_cols]),   men[y_col]
    Xf, yf = add_const(women[x_cols]), women[y_col]
    bm = sm.OLS(ym, Xm, missing="drop").fit().params
    bf = sm.OLS(yf, Xf, missing="drop").fit().params

    prof = {c: float(sub[c].mean()) for c in x_cols if c != "HR"}
    hr_min, hr_max = np.percentile(sub["HR"].dropna(), [5, 95])
    hr_grid = np.linspace(hr_min, hr_max, 100)

    def design(hr_values, coef_index):
        rows = [{"const": 1.0, **prof, "HR": float(h)} for h in hr_values]
        return pd.DataFrame(rows)[coef_index]

    yhat_m = design(hr_grid, bm.index) @ bm
    yhat_f = design(hr_grid, bf.index) @ bf

    fig, ax = plt.subplots(figsize=(8, 6))
    x_pad = 0.05 * (hr_max - hr_min) if hr_max > hr_min else 1.0
    ax.plot(hr_grid, yhat_m, color=COLOR_MEN,   linewidth=2.5, label="Men fit")
    ax.plot(hr_grid, yhat_f, color=COLOR_WOMEN, linewidth=2.5, label="Women fit")
    ax.set_xlabel("Admission heart rate", fontsize=20)
    ax.set_ylabel("Probability of in-hospital mortality", fontsize=20)
    ax.set_xlim(hr_min - x_pad, hr_max + x_pad)
    ax.set_ylim(0.06, 0.20)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=20, loc="upper left", frameon=False)
    ax.tick_params(axis="both", which="major", labelsize=20)
    plt.tight_layout()
    if savepath is not None:
        os.makedirs(os.path.dirname(savepath) or ".", exist_ok=True)
        plt.savefig(savepath, format="pdf", bbox_inches="tight", dpi=300)
    plt.show()


def plot_hr_quartile2_y_hist(
    df_clean, y_col="In-hospital_death", female_col="female",
    target_bin=2, savepath=None,
):
    """Appendix Figure 2: 0/1 mortality counts by gender within HR quartile 2."""
    df = df_clean.copy()
    df["hr_bin"] = pd.qcut(df["HR"], 4, labels=False, duplicates="drop")
    sub = df[df["hr_bin"] == target_bin].copy()

    men   = sub[sub[female_col] == 0][y_col].dropna()
    women = sub[sub[female_col] == 1][y_col].dropna()
    men_counts = men.value_counts().reindex([0, 1], fill_value=0).values
    wom_counts = women.value_counts().reindex([0, 1], fill_value=0).values

    fig, ax = plt.subplots(figsize=(8, 6))
    x = np.array([0, 1], dtype=float)
    width, gap = 0.3, 0.03
    ax.bar(x - (width/2 + gap), men_counts, width=width, color=COLOR_MEN,   label="Men")
    ax.bar(x + (width/2 + gap), wom_counts, width=width, color=COLOR_WOMEN, label="Women")
    ax.set_xlabel("In-hospital death (0/1)", fontsize=20)
    ax.set_ylabel("Count", fontsize=20)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["0", "1"], fontsize=20)
    ax.set_ylim(0, 510)
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=20, loc="upper left", frameon=False)
    ax.tick_params(axis="both", which="major", labelsize=20)
    plt.tight_layout()
    if savepath is not None:
        os.makedirs(os.path.dirname(savepath) or ".", exist_ok=True)
        plt.savefig(savepath, format="pdf", bbox_inches="tight", dpi=300)
    plt.show()


plot_hr_quartile2_lines(
    df_clean=df_clean,
    x_cols=X_COLS,
    savepath="Figures/hr_quartile2_mortality_lines.pdf",
)

plot_hr_quartile2_y_hist(
    df_clean=df_clean,
    savepath="Figures/hr_quartile2_mortality_y_hist.pdf",
)


In [ ]:
# ============================================================
# PART B — All 139 subsets, all 4 models: flip-summary table
# ============================================================
all_rows = []
N = len(subsets)
for j, (label, sub) in enumerate(subsets, start=1):
    print(f"[{j}/{N}] {label} | n={len(sub)}")
    # linear
    out = bootstrap_oaxaca_linear(sub, Y_COL, X_COLS, GROUP_COL,
                                  B=CONFIG["B_linear"], random_state=j,
                                  n_jobs=CONFIG["bootstrap_n_jobs"])
    all_rows.append(build_boot_row("linear", label, len(sub), out))
    # ml
    for mname, mbuilder in ML_BUILDERS.items():
        out = bootstrap_nonlinear(sub, Y_COL, X_COLS, GROUP_COL, mbuilder,
                                  B=CONFIG["B_ml"], random_state=1000 + j,
                                  n_jobs=CONFIG["bootstrap_n_jobs"])
        all_rows.append(build_boot_row(mname, label, len(sub), out))
    pd.DataFrame(all_rows).to_csv("all_subsets_bootstrap_originalnnxgboost.csv", index=False)

all_df = pd.DataFrame(all_rows)
all_df.to_csv("all_subsets_bootstrap_originalnnxgboost.csv", index=False)
print(f"\nDone. {len(all_df)} rows across {N} subsets and {1+len(ML_BUILDERS)} models.")


In [ ]:
# ============================================================
# SECTION 5 — Flip-summary tables (three outputs)
#
#   (A) APPENDIX  — Explained-component flips per model
#   (B) APPENDIX  — Unexplained-component flips per model
#   (C) MAIN TEXT — # of subsets exhibiting a sign flip (subset-wise:
#                   a subset that flips in BOTH components is counted once)
#
# Flip: sign differs between men-ref and women-ref for that component.
# Significant at alpha: at least one side has p < alpha.
# ============================================================

def _component_masks(df, comp):
    men = df[f"{comp}_menref"]; women = df[f"{comp}_womenref"]
    pm  = df[f"{comp}_menref_pvalue"]; pw = df[f"{comp}_womenref_pvalue"]
    is_flip = (men.notna() & women.notna() & (np.sign(men) != np.sign(women))).fillna(False)
    sig10 = (is_flip & ((pm < 0.10) | (pw < 0.10))).fillna(False)
    sig5  = (is_flip & ((pm < 0.05) | (pw < 0.05))).fillna(False)
    sig1  = (is_flip & ((pm < 0.01) | (pw < 0.01))).fillna(False)
    return is_flip, sig10, sig5, sig1

MODEL_ORDER   = ["linear", "logistic", "nn", "xgboost"]
MODEL_DISPLAY = {"linear":"Linear", "logistic":"Logistic",
                 "nn":"Neural net", "xgboost":"XGBoost"}

rows_exp, rows_unexp, rows_subset = [], [], []
for m in MODEL_ORDER:
    s = all_df[all_df["model"] == m]
    if s.empty:
        continue
    e_flip, e10, e5, e1 = _component_masks(s, "explained")
    u_flip, u10, u5, u1 = _component_masks(s, "unexplained")

    # (A) Explained only
    rows_exp.append({"Model": MODEL_DISPLAY[m],
                     "Total flips": int(e_flip.sum()),
                     "10%": int(e10.sum()), "5%": int(e5.sum()), "1%": int(e1.sum())})
    # (B) Unexplained only
    rows_unexp.append({"Model": MODEL_DISPLAY[m],
                       "Total flips": int(u_flip.sum()),
                       "10%": int(u10.sum()), "5%": int(u5.sum()), "1%": int(u1.sum())})
    # (C) Subset-wise: at least one of the two components flips
    any_flip = (e_flip.values | u_flip.values)
    any_10   = (e10.values   | u10.values)
    any_5    = (e5.values    | u5.values)
    any_1    = (e1.values    | u1.values)
    rows_subset.append({"Model": MODEL_DISPLAY[m],
                        "Total flips": int(any_flip.sum()),
                        "10%": int(any_10.sum()), "5%": int(any_5.sum()), "1%": int(any_1.sum())})

explained_summary_df   = pd.DataFrame(rows_exp)
unexplained_summary_df = pd.DataFrame(rows_unexp)
subset_summary_df      = pd.DataFrame(rows_subset)

print("APPENDIX: Explained component flips")
display(explained_summary_df)
print("\nAPPENDIX: Unexplained component flips")
display(unexplained_summary_df)
print("\nMAIN TEXT: Subsets with any sign flip")
display(subset_summary_df)

explained_summary_df.to_csv("flip_summary_explained.csv", index=False)
unexplained_summary_df.to_csv("flip_summary_unexplained.csv", index=False)
subset_summary_df.to_csv("flip_summary_subsetwise.csv", index=False)
